# 🗳️ Proyecto: Análisis Electoral Colombia 2026
Este notebook contiene las herramientas base para realizar web scraping y estructuración de datos electorales, siguiendo la metodología de **FiveThirtyEight**.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime
import os

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}
print('✅ Librerías y configuración listas.')

✅ Librerías y configuración listas.


## 1. Scraper de Noticias (HTML)
Ideal para portales que publican tablas de resultados o encuestas en texto.

In [ ]:
def scrape_tabla_noticias(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        
        # Usamos pandas para leer tablas directamente si existen
        tablas = pd.read_html(response.text)
        if tablas:
            df = tablas[0]
            df['fecha_captura'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            return df
        return 'No se encontraron tablas.'
    except Exception as e:
        return f'Error: {e}'

## 2. Captura de Datos Oficiales (JSON)
La mayoría de mapas de la Registraduría consumen archivos JSON. Scrapear el JSON es 10 veces más preciso que el HTML.

In [ ]:
def get_registraduria_data(json_url):
    try:
        response = requests.get(json_url, headers=HEADERS)
        data = response.json()
        # Aquí procesarías la estructura específica del JSON oficial
        return data
    except Exception as e:
        return f'Error al obtener JSON: {e}'

## 3. Normalización (El 'House Effect')
Estandarizamos nombres para que tu modelo no crea que 'Fico' y 'Federico Gutiérrez' son personas distintas.

In [ ]:
mapeo_nombres = {
    'Fico': 'Federico Gutiérrez',
    'F. Gutiérrez': 'Federico Gutiérrez',
    'Petro': 'Gustavo Petro',
    'G. Petro': 'Gustavo Petro'
}

def limpiar_dataset(df, columna_candidato):
    df[columna_candidato] = df[columna_candidato].replace(mapeo_nombres)
    return df

: 